# Voice Agents: all 45 public methods in azure-ai-projects 2.7.0

## 1. Purpose and safety boundary

This teaching notebook implements the entire Voice Agents method inventory in [the SDK guide](../docs/azure-ai-projects.md): **14 conversation methods + 1 realtime method + 14 telephony methods + 16 methods on returned realtime objects = 45**. Containers, properties, and Python context-manager dunder methods are not extra methods.

**Default Run All is offline.** It imports only Python standard-library modules, defines scenarios, and prints which branches are disabled. It does not import Azure, OpenAI, websockets, aiohttp, or aiortc; inspect credentials/environment variables; access the network; open media files; create threads; or create resources. No installation commands are executed. All live operations are behind the final orchestrator.

**Live means real usage, real stored audio, and potentially real phone charges.** The notebook creates its own uniquely named disposable agent, uses real service-returned IDs, and never accepts an existing agent/conversation/binding/call ID for mutation. Use nonsensitive speech and dedicated test phone resources with all participants' permission. Re-running the live orchestrator creates a new run, not a replay of the previous run.

See [requirements.txt](../requirements.txt) for the repository's current pins. Choose your own Python 3.10+ kernel with **`azure-ai-projects==2.7.0`, `openai>=3.0.0`, and `azure-identity`** actually installed. Sync realtime needs `websockets>=13.0` (provided by the SDK's optional `voice` extras; those extras also include `aiohttp` for async). Optional headless WebRTC/avatar branches additionally use **`aiortc==1.14.0`**. This notebook neither creates an environment nor changes repository dependencies.

Implementation is grounded in the [2.7.0 release notes](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/CHANGELOG.md), not an assumption about the installed SDK. A runtime version guard runs **only after live opt-in**.

## 2. Module and method coverage

Each row has an implemented SDK call, not just an inventory entry. Optional rows are reachable when the corresponding prerequisites and flags are satisfied. A disabled branch is **not** reported as exercised.

| Module / returned object | Distinct methods |
| --- | ---: |
| `client.beta.voice_agents.conversations` | 14 |
| `client.beta.voice_agents.realtime` | 1 |
| `client.beta.voice_agents.telephony` | 14 |
| `manager` | 1 |
| `conn` | 3 |
| `conn.session` | 2 |
| `conn.input_audio_buffer` | 3 |
| `conn.output_audio_buffer` | 1 |
| `conn.conversation.item` | 4 |
| `conn.response` | 2 |
| **Total** | **45** |

### Conversation methods (14)

All paths in this table are under `client.beta.voice_agents.conversations`.

| Method | Implemented scenario | Opt-in |
| --- | --- | --- |
| `delete` | Delete owned persisted conversations in cleanup, including audio cascades | Base live |
| `download_audio` | Consume the merged WAV in bounded memory | Base live |
| `download_audio_item` | Consume the completed assistant turn's WAV | Base live |
| `download_generated_audio_item` | Consume genuine surplus audio after cancellation/truncation | Real audio |
| `get` | Wait for owned conversation finalization | Base live |
| `get_audio` | Read merged-recording metadata before downloading | Base live |
| `get_audio_item` | Read metadata for an actual audio-bearing item | Base live |
| `get_generated_audio_item` | Wait for the interrupted item's generated-audio artifact | Real audio |
| `get_item` | Retrieve the actual completed assistant item | Base live |
| `get_response` | Retrieve the exact completed response captured from events | Base live |
| `list` | List only the disposable agent's conversations | Base live |
| `list_items` | Read its persisted transcript/items | Base live |
| `list_response_items` | Read canonical paged response output | Base live |
| `list_responses` | Read persisted inference turns | Base live |

### Realtime entry point and returned objects (17)

| Method | Implemented scenario | Opt-in |
| --- | --- | --- |
| `client.beta.voice_agents.realtime.connect` | Prepare an owned, timed WebSocket session | Base live |
| `manager.enter` | Explicitly open the manager; pair with `finally` | Base live |
| `conn.close` | Explicit close before persistence reads | Base live |
| `conn.recv` | Bounded typed-event receiver | Base live |
| `conn.send` | Send a typed item-retrieval event; also real WebRTC signaling | Base live |
| `conn.session.update` | Update and verify effective session settings | Base live |
| `conn.session.avatar_connect` | Negotiate using a genuine aiortc SDP offer | Avatar |
| `conn.input_audio_buffer.append` | Send user-supplied PCM speech, never synthetic silence | Real audio |
| `conn.input_audio_buffer.clear` | Discard the first buffered chunk, then resend | Real audio |
| `conn.input_audio_buffer.commit` | Commit with VAD explicitly disabled | Real audio |
| `conn.output_audio_buffer.clear` | Clear real WebRTC playback after cancellation | WebRTC audio |
| `conn.conversation.item.create` | Create a scratch item and a real user prompt | Base live |
| `conn.conversation.item.delete` | Delete only the acknowledged scratch item | Base live |
| `conn.conversation.item.retrieve` | Await retrieval acknowledgement for that item | Base live |
| `conn.conversation.item.truncate` | Mark an actual generated audio item as unheard | Real audio |
| `conn.response.create` | Generate a real response and check terminal status | Base live |
| `conn.response.cancel` | Cancel a response observed in progress; await its terminal event | Real audio / WebRTC audio |

### Telephony methods (14)

All paths in this table are under `client.beta.voice_agents.telephony`.

| Method | Implemented scenario | Opt-in |
| --- | --- | --- |
| `cancel_call_job` | Cancel a distinct, future-scheduled job with no attempts | Outbound calls |
| `create_binding` | Create an owned Twilio binding to an existing dedicated resource | Telephony |
| `create_call_job` | Create the cancellation job and a separate immediate one-attempt job | Outbound calls |
| `delete_binding` | Delete only the binding created by this run, with its ETag | Telephony |
| `end_call` | End the second fresh active inbound test call | Inbound call control |
| `get_binding` | Read the created binding and ETag | Telephony |
| `get_call` | Recheck active/terminal state on owned test calls | Inbound call control |
| `get_call_job` | Observe job state and fresh ETag | Outbound calls |
| `get_transfer_targets` | Read the disposable agent's targets and ETag | Telephony |
| `list_bindings` | List the disposable agent's binding | Telephony |
| `list_calls` | List its call history; discover genuine inbound IDs | Telephony |
| `replace_transfer_targets` | Configure and later clear only this run's target set | Telephony |
| `transfer_call` | Transfer the first fresh active inbound call to the controlled target | Inbound call control |
| `update_binding` | Change only the created binding's label with optimistic concurrency | Telephony |

Agent creation/deletion through `client.agents` is supporting setup, not part of the 45-method count. Outbound jobs do not expose a call ID in the 2.7.0 model, and the call-management routes document **inbound** calls. This notebook does not invent that missing relationship.

## 3. Configuration and independent opt-ins

| Flag | Additional prerequisites / consequences |
| --- | --- |
| `RUN_LIVE_DEMO` | Exact SDK version; `AZURE_AI_PROJECT_ENDPOINT`; prior `az login` or equivalent `DefaultAzureCredential` authentication; Voice Agents preview access; an available managed voice model; `CONFIRM_STORED_AUDIO`. Creates/stores real speech, even for typed input. Foundry-managed storage is necessary for the three download routes. |
| `RUN_REAL_AUDIO` | A real, nonsilent, consented 0.5-15 second WAV: PCM16, mono, 24 kHz. Read-only input; no microphone dependency. Enables input buffers, interrupted response, truncation, and generated-audio retrieval. |
| `RUN_WEBRTC_AUDIO` | aiortc 1.14.0, service WebRTC support, working ICE/UDP/TURN connectivity. Headless receiver creates a real SDP, receives a real audio frame, then clears the server playback buffer. No speakers required. |
| `RUN_AVATAR` | aiortc 1.14.0, avatar-enabled project/model/region, an available character and style, and working ICE connectivity. Receives an actual video frame; does not render a UI. |
| `RUN_TELEPHONY` | Existing configured Twilio Foundry connection and dedicated unused authorized Twilio number; controlled transfer number; `CONFIRM_DEDICATED_PHONE_RESOURCE`. Creates no provider infrastructure and changes no provider webhook itself. |
| `RUN_OUTBOUND_CALLS` | Telephony flag, controlled destination, `CONSENT_TO_BILLABLE_CALLS`. Two distinct durable jobs: future-scheduled/cancelled and immediate. Recipient must answer and hang up the immediate call within the bound. |
| `RUN_INBOUND_CALL_CONTROL` | Telephony flag, controlled caller number, `CONSENT_TO_BILLABLE_CALLS`, `INBOUND_OPERATOR_READY`. Operator temporarily routes the dedicated number to the printed binding webhook, then places **two separate inbound calls** when prompted: first transfer, second hangup. |

All flags and consent switches start `False`. Enabling a child branch without its parent is an error, even if the master live flag is off. No `.env` file is read. Supply endpoint/authentication to your chosen kernel yourself. Do not save real phone numbers, ICE credentials, endpoints, or speech outputs in a committed notebook.

`WEBRTC_ICE_SERVERS` is an optional in-memory list of dictionaries with the aiortc fields `urls`, `username`, and `credential`. With an empty list, only direct host candidates are used: no third-party default STUN service is selected. Avatar ICE settings returned by the service take precedence. Obtain any required TURN configuration outside this notebook; do not provision it here.

In [ ]:
import asyncio
import concurrent.futures
import hashlib
import io
import os
import re
import threading
import time
import wave
from contextlib import ExitStack, contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from importlib.metadata import PackageNotFoundError, version
from itertools import islice
from urllib.parse import urlparse
from uuid import uuid4

SDK_VERSION = '2.7.0'
RUN_LIVE_DEMO = False
RUN_REAL_AUDIO = False
RUN_WEBRTC_AUDIO = False
RUN_AVATAR = False
RUN_TELEPHONY = False
RUN_OUTBOUND_CALLS = False
RUN_INBOUND_CALL_CONTROL = False

CONFIRM_STORED_AUDIO = False
CONFIRM_DEDICATED_PHONE_RESOURCE = False
CONSENT_TO_BILLABLE_CALLS = False
INBOUND_OPERATOR_READY = False

VOICE_MODEL = 'gpt-realtime'
VOICE_NAME = 'en-US-AvaNeural'
REAL_AUDIO_WAV = ''
AVATAR_CHARACTER = ''
AVATAR_STYLE = ''
WEBRTC_ICE_SERVERS = []
TELEPHONY_CONNECTION_NAME = ''
TELEPHONY_PHONE_NUMBER = ''
TRANSFER_PHONE_NUMBER = ''
OUTBOUND_DESTINATION_NUMBER = ''
INBOUND_CALLER_NUMBER = ''

HTTP_TIMEOUT_S = 20
EVENT_TIMEOUT_S = 60
PERSISTENCE_TIMEOUT_S = 120
MEDIA_TIMEOUT_S = 30
CALL_WAIT_TIMEOUT_S = 180
JOB_WAIT_TIMEOUT_S = 180
POLL_INTERVAL_S = 2
MAX_PAGE_ITEMS = 100
MAX_EVENTS = 10000
MAX_AUDIO_BYTES = 16 * 1024 * 1024
PCM_RATE = 24000
LAST_LIVE_RUN = None


## 4. Fail before creating resources when prerequisites are missing

Version/package/environment/file checks below are function bodies, not eager notebook actions. Real-audio validation reads only the explicitly selected input WAV and rejects incompatible/truncated/silent data. It neither generates dummy audio nor converts files behind your back.

In [ ]:
def branch_flags():
    return {
        'real audio': RUN_REAL_AUDIO,
        'WebRTC audio': RUN_WEBRTC_AUDIO,
        'avatar': RUN_AVATAR,
        'telephony configuration': RUN_TELEPHONY,
        'outbound phone calls': RUN_OUTBOUND_CALLS,
        'inbound transfer/end': RUN_INBOUND_CALL_CONTROL,
    }


def validate_branch_dependencies():
    if any(branch_flags().values()) and not RUN_LIVE_DEMO:
        raise ValueError('A live branch is enabled. Explicitly enable RUN_LIVE_DEMO or disable that branch.')
    if (RUN_OUTBOUND_CALLS or RUN_INBOUND_CALL_CONTROL) and not RUN_TELEPHONY:
        raise ValueError('Phone-call branches require RUN_TELEPHONY=True and its dedicated-resource prerequisites.')


def required_text(value, name):
    if not isinstance(value, str) or not value.strip() or '<' in value or '>' in value:
        raise ValueError(f'Configure {name} with a real value before enabling this live branch.')
    return value.strip()


def required_phone(value, name):
    value = required_text(value, name)
    if re.fullmatch(r'\+[1-9][0-9]{7,14}', value) is None:
        raise ValueError(f'{name} must be a controlled international E.164 phone number, including +.')
    return value


def installed_version(distribution):
    try:
        return version(distribution)
    except PackageNotFoundError as exc:
        raise RuntimeError(
            f'The chosen live kernel lacks {distribution}. Select a prepared kernel; this notebook installs nothing.'
        ) from exc


def read_real_pcm():
    path = required_text(REAL_AUDIO_WAV, 'REAL_AUDIO_WAV')
    with wave.open(path, 'rb') as audio:
        if (audio.getnchannels(), audio.getsampwidth(), audio.getframerate(), audio.getcomptype()) != (1, 2, PCM_RATE, 'NONE'):
            raise ValueError('REAL_AUDIO_WAV must be uncompressed mono PCM16 at 24000 Hz; prepare a compatible recording yourself.')
        frames = audio.getnframes()
        if not PCM_RATE // 2 <= frames <= PCM_RATE * 15:
            raise ValueError('Supply 0.5-15 seconds of real, consented speech.')
        pcm = audio.readframes(frames)
    if len(pcm) != frames * 2 or not any(pcm):
        raise ValueError('The supplied WAV is truncated or entirely zero-valued; record real speech, not a dummy buffer.')
    return pcm


def preflight_live():
    actual = installed_version('azure-ai-projects')
    if actual != SDK_VERSION:
        raise RuntimeError(
            f'This notebook requires azure-ai-projects=={SDK_VERSION}; kernel has {actual}. '
            'Choose a kernel matching this notebook\'s SDK contract; manifest edits do not install packages automatically.'
        )
    openai_version = installed_version('openai')
    stable = re.fullmatch(r'(\d+)\.(\d+)\.(\d+)', openai_version)
    if stable is None or tuple(map(int, stable.groups())) < (3, 0, 0):
        raise RuntimeError('The 2.7.0 SDK requires OpenAI >=3.0.0. Select a compatible stable OpenAI release in this kernel.')
    installed_version('azure-identity')
    websocket_version = installed_version('websockets')
    websocket_release = re.fullmatch(r'(\d+)\.(\d+)(?:\.(\d+))?', websocket_version)
    if websocket_release is None or int(websocket_release.group(1)) < 13:
        raise RuntimeError('The SDK voice extra requires websockets>=13.0. Choose a compatible stable version in this kernel.')
    if RUN_WEBRTC_AUDIO or RUN_AVATAR:
        if installed_version('aiortc') != '1.14.0':
            raise RuntimeError('The optional headless media adapter is pinned to aiortc==1.14.0; select a compatible kernel.')
        for server in WEBRTC_ICE_SERVERS:
            if not isinstance(server, dict) or not server.get('urls') or set(server) - {'urls', 'username', 'credential'}:
                raise ValueError('Each WEBRTC_ICE_SERVERS entry needs real urls and only optional username/credential fields.')
    endpoint = required_text(os.environ.get('AZURE_AI_PROJECT_ENDPOINT'), 'AZURE_AI_PROJECT_ENDPOINT in the kernel environment')
    parsed = urlparse(endpoint)
    if parsed.scheme != 'https' or not parsed.hostname or '/api/projects/' not in parsed.path:
        raise ValueError('AZURE_AI_PROJECT_ENDPOINT must be the HTTPS Foundry project endpoint, including /api/projects/<project>.')
    if not CONFIRM_STORED_AUDIO:
        raise ValueError('Set CONFIRM_STORED_AUDIO=True only after consenting to store=True: transcripts AND raw audio persist in Foundry.')
    required_text(VOICE_MODEL, 'VOICE_MODEL')
    required_text(VOICE_NAME, 'VOICE_NAME')
    if RUN_AVATAR:
        required_text(AVATAR_CHARACTER, 'AVATAR_CHARACTER available in this project/region')
        required_text(AVATAR_STYLE, 'AVATAR_STYLE available for that character')
    if RUN_TELEPHONY:
        if not CONFIRM_DEDICATED_PHONE_RESOURCE:
            raise ValueError('Confirm a dedicated, unused test phone resource; existing production bindings/routing must not be reused.')
        required_text(TELEPHONY_CONNECTION_NAME, 'existing TELEPHONY_CONNECTION_NAME')
        required_phone(TELEPHONY_PHONE_NUMBER, 'TELEPHONY_PHONE_NUMBER')
        required_phone(TRANSFER_PHONE_NUMBER, 'TRANSFER_PHONE_NUMBER')
        if TELEPHONY_PHONE_NUMBER == TRANSFER_PHONE_NUMBER:
            raise ValueError('The transfer destination must not loop back to the agent phone number.')
    if RUN_OUTBOUND_CALLS or RUN_INBOUND_CALL_CONTROL:
        if not CONSENT_TO_BILLABLE_CALLS:
            raise ValueError('Explicit CONSENT_TO_BILLABLE_CALLS=True is required: real calls/transfers can incur charges.')
    if RUN_OUTBOUND_CALLS:
        required_phone(OUTBOUND_DESTINATION_NUMBER, 'OUTBOUND_DESTINATION_NUMBER')
        if OUTBOUND_DESTINATION_NUMBER == TELEPHONY_PHONE_NUMBER:
            raise ValueError('Do not originate an outbound call back into this demo binding.')
    if RUN_INBOUND_CALL_CONTROL:
        required_phone(INBOUND_CALLER_NUMBER, 'INBOUND_CALLER_NUMBER')
        if not INBOUND_OPERATOR_READY:
            raise ValueError('An operator must be ready to route the dedicated number, make two inbound calls, and restore routing afterward.')
    pcm = read_real_pcm() if RUN_REAL_AUDIO else None
    from azure.ai.projects import models
    from websockets.sync.client import connect as websocket_connect
    if not callable(websocket_connect):
        raise RuntimeError('The chosen websockets package does not expose the required synchronous client.')
    return endpoint, models, pcm


## 5. Ownership, bounded persistence, and typed-event handling

The ownership ledger stays in `LAST_LIVE_RUN` after errors for manual recovery; nothing is written to disk. Register cleanup immediately after each resource creation. `ExitStack` attempts the remaining callbacks even if one fails and preserves exception chaining. Dependent resources are deliberately retained if an active call/job or earlier cleanup is unresolved.

Persistence polling retries **only** a newly created artifact's 404 or the documented `recording_not_ready` 409. Other 409s, authentication failures, transport errors, failed conversations, failed/incomplete responses, and unrequested cancellations are errors. A generated-audio 404 is not treated as success or replaced with another item's ID. Page/event/audio sizes and waits are bounded.

In [ ]:
@dataclass
class OwnedRun:
    agent_name: str
    agent_version: str | None = None
    conversations: set = field(default_factory=set)
    deleted_conversations: set = field(default_factory=set)
    conversation_discovery_complete: bool = False
    bindings: set = field(default_factory=set)
    jobs: set = field(default_factory=set)
    job_history: dict = field(default_factory=dict)
    job_keys: list = field(default_factory=list)
    calls: set = field(default_factory=set)
    call_history: dict = field(default_factory=dict)
    unexpected_calls: set = field(default_factory=set)
    ownership_conflicts: list = field(default_factory=list)
    target_signature: tuple | None = None


@dataclass
class RealtimeState:
    owned: OwnedRun
    conversation_id: str | None = None
    session: object = None
    response_status: dict = field(default_factory=dict)
    cancelled_on_purpose: set = field(default_factory=set)
    audio_bytes: dict = field(default_factory=dict)
    cleared_responses: set = field(default_factory=set)
    completed_response_id: str | None = None
    completed_audio_item_id: str | None = None
    generated_audio_item_id: str | None = None
    event_count: int = 0


def limited_items(pager, label):
    items = list(islice(pager, MAX_PAGE_ITEMS + 1))
    if len(items) > MAX_PAGE_ITEMS:
        raise RuntimeError(f'{label} exceeded the {MAX_PAGE_ITEMS}-item safety bound; inspect the dedicated run manually.')
    return items


def pause_before_retry(deadline, label):
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError(f'{label}: deadline exceeded; no success or fallback was inferred.')
    time.sleep(min(POLL_INTERVAL_S, remaining))


def wait_persisted(label, read, ready=lambda value: True):
    from azure.core.exceptions import HttpResponseError
    deadline = time.monotonic() + PERSISTENCE_TIMEOUT_S
    while time.monotonic() < deadline:
        try:
            value = read()
        except HttpResponseError as exc:
            detail = getattr(getattr(exc, 'model', None), 'error', None)
            code = getattr(detail, 'code', None)
            if exc.status_code != 404 and not (exc.status_code == 409 and code == 'recording_not_ready'):
                raise
            print(f'{label}: awaiting new artifact persistence (HTTP {exc.status_code}, code={code}).')
            if time.monotonic() >= deadline:
                raise TimeoutError(f'{label} was not persisted within {PERSISTENCE_TIMEOUT_S}s.') from exc
        else:
            if ready(value):
                return value
        pause_before_retry(deadline, label)
    raise TimeoutError(f'{label} was not persisted within {PERSISTENCE_TIMEOUT_S}s.')


def conversation_finished(conversation, for_cleanup=False):
    if conversation.status == 'failed':
        if for_cleanup:
            print('Deleting failed owned conversation; finalization error:', conversation.last_error)
            return True
        raise RuntimeError(f'Conversation finalization failed: {conversation.last_error}')
    if conversation.status not in ('in_progress', 'completed'):
        raise RuntimeError(f'Unknown conversation state: {conversation.status}')
    return conversation.status == 'completed'


def with_etag(pipeline_response, result, response_headers):
    etag = response_headers.get('ETag')
    if not etag:
        raise RuntimeError('The service omitted the ETag required for a conditional telephony mutation; do not use a wildcard update.')
    return result, etag


def require_no_active_telephony(owned):
    if owned.jobs or owned.calls or owned.unexpected_calls:
        raise RuntimeError(
            f'Cleanup cannot remove dependent resources while jobs={sorted(owned.jobs)} or calls={sorted(owned.calls)} '
            f'or unexpected calls={sorted(owned.unexpected_calls)} remain unresolved. '
            'Hang up test phones and reconcile these IDs before manual cleanup.'
        )


def verify_quiet_telephony(client, owned):
    require_no_active_telephony(owned)
    active = limited_items(client.beta.voice_agents.telephony.list_calls(
        agent_name=owned.agent_name, status='in_progress', limit=100,
    ), 'pre-cleanup active-call check')
    if active:
        owned.unexpected_calls.update(call.id for call in active)
        raise RuntimeError('An active call remains on the owned agent; retaining configuration rather than interrupting an untracked caller.')


In [ ]:
def record_conversation(state, conversation_id):
    if not conversation_id:
        raise RuntimeError('store=True did not produce a conversation ID; inspect preview/storage support rather than inventing an ID.')
    state.owned.conversations.add(conversation_id)
    if state.conversation_id is not None and state.conversation_id != conversation_id:
        raise RuntimeError('The service changed the conversation ID within one owned session; stop rather than mixing artifacts.')
    state.conversation_id = conversation_id


def receive_event(conn, state, models, deadline):
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('Realtime event deadline exceeded; the session will be closed by finally.')
    event = conn.recv(timeout=remaining)
    state.event_count += 1
    if state.event_count > MAX_EVENTS:
        raise RuntimeError('Realtime event safety bound exceeded.')
    event_type = event.get('type')
    if event_type in ('error', 'rtc.call.error') or (isinstance(event_type, str) and event_type.endswith('.failed')):
        raise RuntimeError(f'Realtime service failure ({event_type}): {event.get("error", event)}')
    if event_type == 'warning':
        print('Realtime service warning:', event)
    if isinstance(event, models.RealtimeServerEventSessionCreated):
        record_conversation(state, event.conversation_id)
        state.session = event.session
    elif isinstance(event, models.RealtimeServerEventSessionUpdated):
        state.session = event.session
    elif isinstance(event, (models.RealtimeServerEventResponseCreated, models.RealtimeServerEventResponseDone)):
        response = event.response
        required_text(response.id, 'service-returned response ID')
        record_conversation(state, response.conversation_id)
        state.response_status[response.id] = response.status
        if isinstance(event, models.RealtimeServerEventResponseDone):
            allowed = response.status == 'completed' or (
                response.status == 'cancelled' and response.id in state.cancelled_on_purpose
            )
            if not allowed:
                raise RuntimeError(f'Response {response.id} ended as {response.status}: {response.status_details}')
    elif isinstance(event, models.RealtimeServerEventResponseAudioDelta):
        key = (event.response_id, event.item_id, event.content_index)
        state.audio_bytes[key] = state.audio_bytes.get(key, 0) + len(event.delta)
        if sum(state.audio_bytes.values()) > MAX_AUDIO_BYTES:
            raise RuntimeError('Generated audio exceeded the per-session byte limit.')
    elif isinstance(event, models.RealtimeServerEventOutputAudioBufferCleared):
        state.cleared_responses.add(event.response_id)
    return event


def wait_event(conn, state, models, label, predicate):
    deadline = time.monotonic() + EVENT_TIMEOUT_S
    while time.monotonic() < deadline:
        event = receive_event(conn, state, models, deadline)
        if predicate(event):
            return event
    raise TimeoutError(f'{label}: no matching acknowledgement within {EVENT_TIMEOUT_S}s.')


def wait_response(conn, state, models, response_id, expected='completed'):
    if state.response_status.get(response_id) == 'in_progress':
        wait_event(
            conn, state, models, 'response.done',
            lambda event: isinstance(event, models.RealtimeServerEventResponseDone) and event.response.id == response_id,
        )
    actual = state.response_status.get(response_id)
    if actual != expected:
        raise RuntimeError(f'Response {response_id}: expected {expected}, observed {actual}. A cancellation race is not success.')


def start_response(conn, state, models, instructions, max_tokens=192):
    conn.response.create(response=models.VoiceAgentResponseCreateParams(
        instructions=instructions, max_output_tokens=max_tokens,
    ))
    event = wait_event(
        conn, state, models, 'response.created',
        lambda value: isinstance(value, models.RealtimeServerEventResponseCreated),
    )
    if event.response.status != 'in_progress':
        raise RuntimeError(f'New response was not active: {event.response.status}')
    return event.response.id


def create_text_item(conn, state, models, text):
    conn.conversation.item.create(item=models.RealtimeConversationItemMessageUser(
        type=models.RealtimeConversationItemType.MESSAGE,
        content=[models.RealtimeConversationItemMessageUserContent(type='input_text', text=text)],
    ))
    event = wait_event(
        conn, state, models, 'user item creation',
        lambda value: isinstance(value, (
            models.RealtimeServerEventConversationItemCreated,
            models.RealtimeServerEventConversationItemAdded,
        )) and value.item.get('role') == 'user'
        and any(part.get('text') == text for part in value.item.get('content', [])),
    )
    return required_text(event.item.id, 'service-returned item ID')


## 6. Disposable agent and a typed realtime text conversation

A UUID-based **new agent name**, not a fabricated service ID, isolates the run. An existence check refuses to touch a pre-existing agent. The exact returned version is recorded before anything else. `store=True` is intentional: without it there are no persisted transcripts or recordings.

The normal SDK idiom is `with client.beta.voice_agents.realtime.connect(...) as conn`. Here `manager.enter()` is explicit so that it is actually demonstrated; a `try/finally` immediately owns `conn.close()`. A scratch text item is created, retrieved both with the helper and with typed `send`, and deleted before generating the real reply. The reply is synthesized audio even though its input is text. No local speaker plays it.

In [ ]:
def create_owned_agent(client, owned, models, cleanup):
    from azure.core.exceptions import ResourceNotFoundError
    try:
        client.agents.get(agent_name=owned.agent_name)
    except ResourceNotFoundError:
        print('New disposable agent name confirmed:', owned.agent_name)
    else:
        raise RuntimeError('The generated agent name already exists; refusing to create a version on somebody else\'s agent.')
    created = client.agents.create_version(
        agent_name=owned.agent_name,
        definition=models.VoiceAgentDefinition(
            model_type=models.VoiceModelType.MANAGED,
            model=VOICE_MODEL,
            instructions='You are a friendly test voice assistant. Keep ordinary replies short. Do not initiate calls or transfers.',
            audio=models.VoiceAgentAudioConfig(output=models.VoiceAgentAudioOutputConfig(
                format=models.RealtimeAudioFormatsAudioPcm(rate=24000),
                voice=VOICE_NAME, voice_type=models.VoiceType.AZURE_STANDARD,
            )),
            output_modalities=[models.VoiceOutputModality.AUDIO],
            store=True,
        ),
    )
    owned.agent_version = created.version
    cleanup.callback(cleanup_agent, client, owned)
    cleanup.callback(cleanup_conversations, client, owned)
    print('Created owned agent version:', created.version)


@contextmanager
def owned_realtime_session(client, owned, models):
    manager = client.beta.voice_agents.realtime.connect(
        agent_name=owned.agent_name,
        open_timeout=HTTP_TIMEOUT_S,
        close_timeout=5,
        max_size=MAX_AUDIO_BYTES,
    )
    conn = manager.enter()
    try:
        state = RealtimeState(owned)
        wait_event(conn, state, models, 'session.created', lambda event: isinstance(event, models.RealtimeServerEventSessionCreated))
        yield conn, state
    finally:
        conn.close(code=1000, reason='Owned notebook scenario finished')


def text_and_optional_audio_scenario(client, owned, models, pcm):
    with owned_realtime_session(client, owned, models) as (conn, state):
        conn.session.update(session=models.VoiceAgentSessionUpdateConfig(
            instructions='This is a disposable SDK demonstration. Answer ordinary questions in one brief sentence.',
            max_output_tokens=192,
        ))
        wait_event(conn, state, models, 'session.updated', lambda event: isinstance(event, models.RealtimeServerEventSessionUpdated))
        scratch_id = create_text_item(conn, state, models, 'Temporary notebook note: retrieve and remove this before answering.')
        conn.conversation.item.retrieve(item_id=scratch_id)
        wait_event(
            conn, state, models, 'helper retrieval',
            lambda event: isinstance(event, models.RealtimeServerEventConversationItemRetrieved) and event.item.id == scratch_id,
        )
        conn.send(models.RealtimeClientEventConversationItemRetrieve(item_id=scratch_id))
        wait_event(
            conn, state, models, 'typed send retrieval',
            lambda event: isinstance(event, models.RealtimeServerEventConversationItemRetrieved) and event.item.id == scratch_id,
        )
        conn.conversation.item.delete(item_id=scratch_id)
        wait_event(
            conn, state, models, 'scratch deletion',
            lambda event: isinstance(event, models.RealtimeServerEventConversationItemDeleted) and event.item_id == scratch_id,
        )
        create_text_item(conn, state, models, 'Please welcome me to this voice SDK notebook in one short sentence.')
        response_id = start_response(conn, state, models, 'Give a single short spoken welcome to the voice SDK notebook.')
        wait_response(conn, state, models, response_id)
        audio_keys = [key for key, size in state.audio_bytes.items() if key[0] == response_id and size > 0]
        if not audio_keys:
            raise RuntimeError('The completed reply contained no real audio; recording methods need an audio-capable model/voice.')
        state.completed_response_id = response_id
        state.completed_audio_item_id = audio_keys[0][1]
        print('Completed real synthesized reply:', response_id)
        if RUN_REAL_AUDIO:
            real_audio_interruption_scenario(conn, state, models, pcm)
        else:
            print('DISABLED: real input audio, interruption/truncation, and generated-audio download branch.')
    return state


## 7. Real audio buffers, response cancellation, and truncation

Disable VAD with an explicit JSON null through the model's mapping interface and verify the effective session configuration before appending. Append a real first chunk, clear it with acknowledgement, resend the recording in bounded chunks, then commit. No guessed item IDs or silence buffers are used.

The interruption waits for real decoded `RealtimeServerEventResponseAudioDelta` bytes and their `response_id`, `item_id`, and `content_index`. It then cancels that active response, confirms `response.done(status='cancelled')`, and truncates its genuine assistant item to **0 ms heard**, because this headless WebSocket branch never plays any audio. This gives the service a truthful heard/generated distinction. The untouched first reply remains available for canonical item/merged recording downloads.

The separate generated-audio artifact exists only if the service retained audio beyond the heard segment. A bounded wait and explicit error are used if that artifact is unavailable. `output_audio_buffer.clear` is deliberately **not** sent on this ordinary WebSocket transport; section 9 establishes real WebRTC for it.

In [ ]:
def real_audio_interruption_scenario(conn, state, models, pcm):
    if not isinstance(pcm, bytes) or len(pcm) < PCM_RATE:
        raise ValueError('Real-audio branch requires the validated user WAV from preflight.')
    input_config = models.VoiceAgentAudioInputConfig(format=models.RealtimeAudioFormatsAudioPcm(rate=24000))
    input_config['turn_detection'] = None
    conn.session.update(session=models.VoiceAgentSessionUpdateConfig(
        audio=models.VoiceAgentAudioConfig(input=input_config),
    ))
    updated = wait_event(
        conn, state, models, 'manual audio turn configuration',
        lambda event: isinstance(event, models.RealtimeServerEventSessionUpdated),
    )
    effective = updated.session.audio
    if effective is None or effective.input is None or effective.input.turn_detection is not None:
        raise RuntimeError('Service did not confirm disabled VAD; manual commit would race automatic turns.')
    conn.input_audio_buffer.append(audio=pcm[:9600])
    conn.input_audio_buffer.clear()
    wait_event(conn, state, models, 'input buffer cleared', lambda event: event.get('type') == 'input_audio_buffer.cleared')
    for start in range(0, len(pcm), 9600):
        conn.input_audio_buffer.append(audio=pcm[start:start + 9600])
    conn.input_audio_buffer.commit()
    committed = wait_event(
        conn, state, models, 'input buffer committed',
        lambda event: isinstance(event, models.RealtimeServerEventInputAudioBufferCommitted),
    )
    print('Committed genuine user-audio item:', committed.item_id)
    response_id = start_response(
        conn, state, models,
        'For this interruption test, count slowly from one to one hundred with pauses between numbers.',
        max_tokens=1024,
    )
    first_audio = wait_event(
        conn, state, models, 'actual audio before cancellation',
        lambda event: (
            isinstance(event, models.RealtimeServerEventResponseAudioDelta)
            and event.response_id == response_id and len(event.delta) > 0
        ) or (
            isinstance(event, models.RealtimeServerEventResponseDone) and event.response.id == response_id
        ),
    )
    if not isinstance(first_audio, models.RealtimeServerEventResponseAudioDelta):
        raise RuntimeError('Response ended before an audio chunk could be interrupted. No generated-audio coverage is claimed.')
    if first_audio.content_index != 0:
        raise RuntimeError('The SDK truncate helper documents content_index=0; this response needs a different scenario.')
    state.cancelled_on_purpose.add(response_id)
    conn.response.cancel(response_id=response_id)
    wait_response(conn, state, models, response_id, expected='cancelled')
    conn.conversation.item.truncate(item_id=first_audio.item_id, content_index=first_audio.content_index, audio_end_ms=0)
    truncated = wait_event(
        conn, state, models, 'actual assistant-audio truncation',
        lambda event: isinstance(event, models.RealtimeServerEventConversationItemTruncated)
        and event.item_id == first_audio.item_id,
    )
    if truncated.audio_end_ms != 0 or truncated.content_index != first_audio.content_index:
        raise RuntimeError('Truncation acknowledgement did not match the requested unheard audio segment.')
    state.generated_audio_item_id = truncated.item_id
    print('Confirmed cancelled response and real unheard audio item:', response_id, truncated.item_id)


## 8. All persisted conversation/audio read methods

Run this phase **after `conn.close()`**. Wait for the known conversation to reach `completed`, read its response and canonical paged output, verify the observed assistant item has `output_audio`, then fetch audio metadata before bytes. List responses may omit `output`, so `list_response_items` is not replaced by a list-result projection.

Download iterators are consumed, not merely constructed. WAV data is checked in bounded memory and reported by size, channels, rate, frame count, and SHA-256; no local recordings are written. The underlying HTTP stream is closed even if the bound or WAV validation fails. Metadata IDs must match the typed-event IDs. `blob_uri` means BYOS: the service download routes do not proxy those bytes, so this notebook raises a clear prerequisite error rather than pretending a download succeeded.

In [ ]:
def verify_audio_metadata(metadata, conversation_id, item_id=None):
    if metadata.conversation_id != conversation_id:
        raise RuntimeError('Audio metadata belongs to a different conversation.')
    if item_id is not None and metadata.item_id != item_id:
        raise RuntimeError('Audio metadata belongs to a different item.')
    if metadata.blob_uri:
        raise RuntimeError(
            'This artifact uses BYOS. The SDK download route cannot proxy it. '
            'Use a Foundry-managed-storage project for this demo; no customer-storage credentials or fallback download are attempted.'
        )
    if metadata.format != 'wav' or metadata.duration_ms is None or metadata.duration_ms.total_seconds() <= 0:
        raise RuntimeError('Metadata does not confirm a nonempty persisted WAV artifact.')


def consume_wav_download(label, fetch, metadata):
    response_holder = {}
    def capture_response(pipeline_response):
        response_holder['response'] = pipeline_response.http_response
    deadline = time.monotonic() + EVENT_TIMEOUT_S
    with ExitStack() as cleanup, io.BytesIO() as buffer:
        chunks = fetch(capture_response)
        if 'response' not in response_holder:
            raise RuntimeError('The download did not expose its HTTP response for deterministic closure.')
        cleanup.callback(response_holder['response'].close)
        for chunk in chunks:
            if time.monotonic() >= deadline:
                raise TimeoutError(f'{label}: download exceeded the time bound.')
            if buffer.tell() + len(chunk) > MAX_AUDIO_BYTES:
                raise RuntimeError(f'{label}: download exceeded {MAX_AUDIO_BYTES} bytes.')
            buffer.write(chunk)
        payload = buffer.getvalue()
    with wave.open(io.BytesIO(payload), 'rb') as audio:
        channels, rate, frames = audio.getnchannels(), audio.getframerate(), audio.getnframes()
        if frames <= 0 or audio.getsampwidth() != 2 or audio.getcomptype() != 'NONE':
            raise RuntimeError(f'{label}: expected a nonempty PCM16 WAV from the configured PCM voice.')
        if channels != metadata.channels or rate != metadata.sample_rate:
            raise RuntimeError(f'{label}: WAV header and service metadata disagree.')
        if len(audio.readframes(frames)) != frames * channels * 2:
            raise RuntimeError(f'{label}: WAV payload is truncated.')
    print(label, {'bytes': len(payload), 'channels': channels, 'sample_rate': rate, 'frames': frames,
                  'sha256': hashlib.sha256(payload).hexdigest()})


def persisted_conversation_scenario(client, owned, state):
    agent_name = owned.agent_name
    conversation_id = required_text(state.conversation_id, 'owned conversation ID')
    response_id = required_text(state.completed_response_id, 'completed response ID')
    item_id = required_text(state.completed_audio_item_id, 'completed assistant audio item ID')
    conversation = wait_persisted(
        'conversation finalization',
        lambda: client.beta.voice_agents.conversations.get(agent_name=agent_name, conversation_id=conversation_id),
        conversation_finished,
    )
    conversations = limited_items(client.beta.voice_agents.conversations.list(agent_name=agent_name, limit=100), 'conversations')
    if conversation_id not in {value.id for value in conversations}:
        raise RuntimeError('The completed owned conversation is missing from the list result.')
    responses = limited_items(client.beta.voice_agents.conversations.list_responses(
        agent_name=agent_name, conversation_id=conversation_id, limit=100, order='asc',
    ), 'responses')
    response = client.beta.voice_agents.conversations.get_response(
        agent_name=agent_name, conversation_id=conversation_id, response_id=response_id,
    )
    if response.id != response_id or response.conversation_id != conversation_id or response.status != 'completed':
        raise RuntimeError('Persisted response does not match the completed realtime response.')
    if response_id not in {value.id for value in responses}:
        raise RuntimeError('The completed response is missing from persisted response history.')
    response_items = limited_items(client.beta.voice_agents.conversations.list_response_items(
        agent_name=agent_name, conversation_id=conversation_id, response_id=response_id, limit=100, order='asc',
    ), 'response items')
    items = limited_items(client.beta.voice_agents.conversations.list_items(
        agent_name=agent_name, conversation_id=conversation_id, limit=100, order='asc',
    ), 'conversation items')
    item = client.beta.voice_agents.conversations.get_item(
        agent_name=agent_name, conversation_id=conversation_id, item_id=item_id,
    )
    if item.id != item_id or item.get('role') != 'assistant' or not any(
        part.get('type') == 'output_audio' for part in item.get('content', [])
    ):
        raise RuntimeError('Typed-event item ID did not resolve to the expected persisted assistant audio item.')
    if item_id not in {value.id for value in items} or item_id not in {value.id for value in response_items}:
        raise RuntimeError('The real audio item is absent from a canonical paged projection.')
    print('Persisted conversation:', conversation.id, 'responses:', len(responses), 'items:', len(items))
    recording = wait_persisted('merged recording metadata', lambda: client.beta.voice_agents.conversations.get_audio(
        agent_name=agent_name, conversation_id=conversation_id,
    ))
    verify_audio_metadata(recording, conversation_id)
    consume_wav_download('Merged recording', lambda hook: client.beta.voice_agents.conversations.download_audio(
        agent_name=agent_name, conversation_id=conversation_id, raw_response_hook=hook,
    ), recording)
    segment = wait_persisted('canonical item audio metadata', lambda: client.beta.voice_agents.conversations.get_audio_item(
        agent_name=agent_name, conversation_id=conversation_id, item_id=item_id,
    ))
    verify_audio_metadata(segment, conversation_id, item_id)
    consume_wav_download('Completed assistant item', lambda hook: client.beta.voice_agents.conversations.download_audio_item(
        agent_name=agent_name, conversation_id=conversation_id, item_id=item_id, raw_response_hook=hook,
    ), segment)
    if RUN_REAL_AUDIO:
        generated_id = required_text(state.generated_audio_item_id, 'acknowledged interrupted assistant item ID')
        if generated_id not in {value.id for value in items}:
            raise RuntimeError('The interrupted item was not persisted; do not substitute the completed item ID.')
        generated = wait_persisted(
            'generated audio beyond the heard segment; verify interruption support if this times out',
            lambda: client.beta.voice_agents.conversations.get_generated_audio_item(
                agent_name=agent_name, conversation_id=conversation_id, item_id=generated_id,
            ),
        )
        verify_audio_metadata(generated, conversation_id, generated_id)
        consume_wav_download('Interrupted generated audio', lambda hook: client.beta.voice_agents.conversations.download_generated_audio_item(
            agent_name=agent_name, conversation_id=conversation_id, item_id=generated_id, raw_response_hook=hook,
        ), generated)


## 9. Genuine WebRTC audio and avatar media (independent opt-ins)

The generated 2.7.0 model explicitly marks `output_audio_buffer.clear` as **WebRTC/SIP only**. A fake SDP or a regular text WebSocket would not demonstrate it. The optional adapter below uses a real aiortc peer, real ICE gathering, and real received frames. It is entirely lazy: no media imports, event loops, sockets, or threads exist in offline mode.

A dedicated asyncio loop keeps the peer alive while the synchronous SDK receives signaling events. Every peer coroutine has a timeout; shutdown closes the peer and stops/joins its thread. The adapter receives and discards media in memory, without devices, a web server, an audio file, or an avatar display. Codec/ICE/service errors propagate.

* WebRTC audio sends the typed `VoiceAgentClientEventRtcCallSdpCreate(sdp_offer=...)` through `conn.send`, applies the actual `rtc.call.sdp.created.sdp_answer`, receives an audio frame, cancels the active response, clears the output buffer, and requires both matching acknowledgements.
* Avatar updates typed avatar settings, uses returned ICE configuration if supplied, calls `conn.session.avatar_connect(client_sdp=...)`, applies `session.avatar.connecting.server_sdp`, and requires a decoded video frame. This is media negotiation/reception, not a claim of rendered UI quality.

These are distinct service capabilities; enabling the avatar branch does not assert that generic WebRTC audio is supported, or vice versa. There is no downgrade to a fake/media-free success.

In [ ]:
class HeadlessPeer:
    def __init__(self):
        self.loop = None
        self.thread = None
        self.pc = None
        self.tracks = {}
        self.started = threading.Event()

    def _run_loop(self):
        asyncio.set_event_loop(self.loop)
        self.started.set()
        try:
            self.loop.run_forever()
        finally:
            self.loop.close()

    def _submit(self, coroutine, label):
        async def bounded():
            return await asyncio.wait_for(coroutine, timeout=MEDIA_TIMEOUT_S)
        future = asyncio.run_coroutine_threadsafe(bounded(), self.loop)
        try:
            return future.result(timeout=MEDIA_TIMEOUT_S + 2)
        except concurrent.futures.TimeoutError as exc:
            future.cancel()
            raise TimeoutError(f'{label} timed out; check ICE/TURN, codec, and preview availability.') from exc

    async def _create_offer(self, kinds, ice_servers):
        from aiortc import RTCConfiguration, RTCIceServer, RTCPeerConnection
        self.pc = RTCPeerConnection(RTCConfiguration(iceServers=[RTCIceServer(**server) for server in ice_servers]))
        @self.pc.on('track')
        def on_track(track):
            self.tracks[track.kind] = track
        for kind in kinds:
            self.pc.addTransceiver(kind, direction='recvonly')
        offer = await self.pc.createOffer()
        await self.pc.setLocalDescription(offer)
        local = self.pc.localDescription
        if local is None or local.type != 'offer' or self.pc.iceGatheringState != 'complete':
            raise RuntimeError('aiortc did not produce an ICE-complete local SDP offer.')
        return local.sdp

    def start(self, kinds, ice_servers):
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self._run_loop, name='voice-notebook-media', daemon=True)
        self.thread.start()
        if not self.started.wait(timeout=5):
            raise TimeoutError('The local WebRTC event loop did not start within 5 seconds.')
        return self._submit(self._create_offer(kinds, ice_servers), 'Real SDP creation')

    async def _apply_answer(self, sdp):
        from aiortc import RTCSessionDescription
        await self.pc.setRemoteDescription(RTCSessionDescription(sdp=sdp, type='answer'))
        while self.pc.connectionState != 'connected':
            if self.pc.connectionState in ('failed', 'closed'):
                raise RuntimeError(f'WebRTC transport ended as {self.pc.connectionState}.')
            await asyncio.sleep(0.1)

    def apply_answer(self, sdp):
        return self._submit(self._apply_answer(required_text(sdp, 'real service SDP answer')), 'WebRTC connection')

    async def _receive_frame(self, kind):
        while kind not in self.tracks:
            if self.pc.connectionState != 'connected':
                raise RuntimeError(f'WebRTC disconnected before a {kind} track arrived.')
            await asyncio.sleep(0.1)
        frame = await self.tracks[kind].recv()
        if kind == 'audio':
            if frame.samples <= 0:
                raise RuntimeError('An empty audio frame is not a playback demonstration.')
            return {'samples': frame.samples, 'sample_rate': frame.sample_rate}
        if frame.width <= 0 or frame.height <= 0:
            raise RuntimeError('An empty video frame is not an avatar demonstration.')
        return {'width': frame.width, 'height': frame.height}

    def receive_frame(self, kind):
        return self._submit(self._receive_frame(kind), f'Real {kind} frame')

    def close(self):
        if self.loop is None:
            return
        try:
            if self.pc is not None:
                self._submit(self.pc.close(), 'WebRTC peer cleanup')
        finally:
            if self.thread is not None and self.thread.is_alive():
                self.loop.call_soon_threadsafe(self.loop.stop)
                self.thread.join(timeout=5)
                if self.thread.is_alive():
                    raise RuntimeError('The media thread did not stop; restart the selected kernel after checking cleanup.')
            elif not self.loop.is_closed():
                self.loop.close()


In [ ]:
def webrtc_audio_scenario(client, owned, models):
    with owned_realtime_session(client, owned, models) as (conn, state), ExitStack() as cleanup:
        peer = HeadlessPeer()
        cleanup.callback(peer.close)
        offer = peer.start(('audio',), WEBRTC_ICE_SERVERS)
        conn.send(models.VoiceAgentClientEventRtcCallSdpCreate(sdp_offer=offer))
        answer = wait_event(
            conn, state, models, 'rtc.call.sdp.created',
            lambda event: isinstance(event, models.VoiceAgentServerEventRtcCallSdpCreated),
        )
        peer.apply_answer(answer.sdp_answer)
        create_text_item(conn, state, models, 'Please count slowly from one to one hundred for a playback interruption test.')
        response_id = start_response(conn, state, models, 'Count slowly from one to one hundred with pauses.', max_tokens=1024)
        print('Received real WebRTC audio frame:', peer.receive_frame('audio'))
        state.cancelled_on_purpose.add(response_id)
        conn.response.cancel(response_id=response_id)
        conn.output_audio_buffer.clear()
        wait_event(
            conn, state, models, 'cancelled response and cleared WebRTC output',
            lambda event: state.response_status.get(response_id) in ('cancelled', 'completed')
            and response_id in state.cleared_responses,
        )
        wait_response(conn, state, models, response_id, expected='cancelled')
        print('Confirmed WebRTC output-buffer clear for response:', response_id)


def avatar_scenario(client, owned, models):
    with owned_realtime_session(client, owned, models) as (conn, state), ExitStack() as cleanup:
        conn.session.update(session=models.VoiceAgentSessionUpdateConfig(
            avatar=models.VoiceAgentSessionAvatarConfig(
                type=models.VoiceAgentAvatarType.VIDEO_AVATAR,
                character=AVATAR_CHARACTER, style=AVATAR_STYLE,
                output_protocol=models.VoiceAgentAvatarOutputProtocol.WEBRTC,
            ),
            output_modalities=[models.VoiceOutputModality.AUDIO, models.VoiceOutputModality.AVATAR],
        ))
        updated = wait_event(
            conn, state, models, 'avatar session configuration',
            lambda event: isinstance(event, models.RealtimeServerEventSessionUpdated),
        )
        avatar = updated.session.avatar
        if avatar is None or avatar.output_protocol != 'webrtc':
            raise RuntimeError('Service did not enable a WebRTC avatar; check character, model, region, and preview access.')
        ice_servers = WEBRTC_ICE_SERVERS
        if avatar.ice_servers:
            ice_servers = [
                {'urls': server.urls, 'username': server.username, 'credential': server.credential}
                for server in avatar.ice_servers
            ]
        peer = HeadlessPeer()
        cleanup.callback(peer.close)
        offer = peer.start(('audio', 'video'), ice_servers)
        conn.session.avatar_connect(client_sdp=offer)
        answer = wait_event(
            conn, state, models, 'session.avatar.connecting',
            lambda event: isinstance(event, models.VoiceAgentServerEventSessionAvatarConnecting),
        )
        peer.apply_answer(answer.server_sdp)
        create_text_item(conn, state, models, 'Say hello briefly for this avatar media test.')
        response_id = start_response(conn, state, models, 'Say one short friendly greeting.')
        print('Received real avatar video frame:', peer.receive_frame('video'))
        wait_response(conn, state, models, response_id)


## 10. Owned telephony binding and transfer targets

This concrete provider scenario uses **Twilio**, not guessed provider-neutral fields. Prerequisites are an existing project connection and an authorized dedicated test number. The notebook creates no Twilio/Teams/ACS infrastructure and does not modify provider routing or credentials. Binding creation returns the webhook URL that an operator can configure on the dedicated test number if inbound call control is selected.

All mutation IDs come directly from this run. Reads use the documented `cls` callback to capture the real `ETag`; updates, deletion, cancellation, and replacement use `MatchConditions.IfNotModified`, never a guessed revision or wildcard update. An unexpected pre-existing transfer-target set is an error. Cleanup refuses to clear targets that no longer match the exact set created here.

In [ ]:
def target_signature(targets):
    return tuple(sorted(
        (target.name, target.description, target.destination.kind, target.destination.get('value'))
        for target in targets
    ))


def telephony_configuration_scenario(client, owned, models, cleanup):
    from azure.core import MatchConditions
    agent_name = owned.agent_name
    binding = client.beta.voice_agents.telephony.create_binding(
        agent_name=agent_name,
        telephony_binding=models.CreateTwilioTelephonyBindingRequest(
            connection_name=TELEPHONY_CONNECTION_NAME,
            phone_number=TELEPHONY_PHONE_NUMBER,
            label='Disposable notebook binding',
        ),
    )
    owned.bindings.add(binding.id)
    cleanup.callback(cleanup_binding, client, owned, binding.id)
    fetched, etag = client.beta.voice_agents.telephony.get_binding(
        agent_name=agent_name, binding_id=binding.id, cls=with_etag,
    )
    if fetched.id != binding.id or fetched.connection_name != TELEPHONY_CONNECTION_NAME:
        raise RuntimeError('Created binding did not resolve to the configured dedicated connection.')
    updated = client.beta.voice_agents.telephony.update_binding(
        agent_name=agent_name, binding_id=binding.id,
        body=models.UpdateTelephonyBindingRequest(label='Disposable notebook binding - checked'),
        etag=etag, match_condition=MatchConditions.IfNotModified,
    )
    if updated.label != 'Disposable notebook binding - checked':
        raise RuntimeError('Binding update was not reflected in the returned resource.')
    bindings = limited_items(client.beta.voice_agents.telephony.list_bindings(agent_name=agent_name, limit=100), 'bindings')
    if binding.id not in {value.id for value in bindings}:
        raise RuntimeError('The newly created binding is absent from its agent listing.')
    current, etag = client.beta.voice_agents.telephony.get_transfer_targets(agent_name=agent_name, cls=with_etag)
    if current.transfer_targets:
        owned.ownership_conflicts.append('Unexpected pre-existing transfer targets on the new agent')
        raise RuntimeError('Unexpected pre-existing transfer targets; refusing to replace them.')
    targets = [models.TelephonyTransferTarget(
        name='notebook-controlled-destination',
        description='Consenting test recipient; only transfer when the notebook operator requests it.',
        destination=models.PSTNTelephonyTransferDestination(value=TRANSFER_PHONE_NUMBER),
    )]
    replaced = client.beta.voice_agents.telephony.replace_transfer_targets(
        agent_name=agent_name, transfer_targets=targets, etag=etag, match_condition=MatchConditions.IfNotModified,
    )
    owned.target_signature = target_signature(targets)
    cleanup.callback(cleanup_transfer_targets, client, owned)
    if target_signature(replaced.transfer_targets) != owned.target_signature:
        raise RuntimeError('The service returned a different transfer-target configuration.')
    calls = limited_items(client.beta.voice_agents.telephony.list_calls(agent_name=agent_name, limit=100), 'call history')
    print('Owned binding:', binding.id, '; initial call history entries:', len(calls))
    if RUN_INBOUND_CALL_CONTROL:
        print('For the dedicated test number ONLY, temporarily configure this incoming-call webhook:')
        print(binding.incoming_call_url)
        print('Restore/remove that provider webhook after this run; the notebook does not edit it.')
    return binding


## 11. Two distinct outbound jobs: cancellation and a real call

Only the SDK's real typed fields are used: `destination`, `connection_name`, `source`, `purpose`, and optional `schedule`. For Twilio, `source` is the authorized E.164 number. Each create request gets a unique recorded idempotency key. Omitting `retry_policy` requests one attempt, not a retry campaign.

1. Schedule one real job one hour in the future, confirm it has no provider attempts and is cancellable, cancel with its fresh ETag, and require terminal `cancelled`.
2. Create a **different** immediate job to the controlled destination. The consenting recipient answers and hangs up within the bound. Require terminal `completed`; blocked/expired/failed/unexpected-cancelled are errors.

**Cancellation is not hangup:** the SDK documents that a connected outbound call is allowed to finish. Cleanup requests cancellation and waits with a bound. If a connected call will not finish, cleanup raises and retains the dependent agent/binding for recovery rather than reporting success. A killed kernel cannot run `finally`; the future job is durable and must be cancelled manually using the recorded ID/key if automatic cleanup never runs.

In [ ]:
JOB_TERMINAL = frozenset({'completed', 'blocked', 'expired', 'failed', 'cancelled'})
JOB_ACTIVE = frozenset({'accepted', 'waiting_for_schedule', 'queued', 'dispatching', 'in_progress',
                        'waiting_for_retry', 'cancellation_requested'})


def observe_job(owned, job):
    owned.job_history[job.id] = job.status
    if job.status in JOB_TERMINAL:
        owned.jobs.discard(job.id)
    elif job.status not in JOB_ACTIVE:
        raise RuntimeError(f'Unknown call-job state {job.status}; inspect job {job.id} manually.')


def wait_job(client, owned, job_id, expected):
    deadline = time.monotonic() + JOB_WAIT_TIMEOUT_S
    while time.monotonic() < deadline:
        job = client.beta.voice_agents.telephony.get_call_job(agent_name=owned.agent_name, call_job_id=job_id)
        observe_job(owned, job)
        if job.status in JOB_TERMINAL:
            if job.status not in expected:
                raise RuntimeError(f'Job {job.id} ended as {job.status}: {job.terminal_reason}')
            print('Call job terminal state:', job.id, job.status)
            return job
        pause_before_retry(deadline, f'Call job {job_id}; connected calls need the recipient to hang up')
    raise TimeoutError(f'Job {job_id} did not finish; cancellation does not terminate an already connected call.')


def submit_owned_job(client, owned, models, cleanup, schedule, purpose):
    key = str(uuid4())
    owned.job_keys.append(key)
    print('Submitting an explicitly consented real outbound job. Recovery idempotency key:', key)
    job = client.beta.voice_agents.telephony.create_call_job(
        agent_name=owned.agent_name,
        idempotency_key=key,
        body=models.CreateTelephonyCallJobRequest(
            destination=models.TelephonyOutboundDestination(type='phone_number', value=OUTBOUND_DESTINATION_NUMBER),
            connection_name=TELEPHONY_CONNECTION_NAME,
            source=TELEPHONY_PHONE_NUMBER,
            purpose=purpose,
            schedule=schedule,
        ),
    )
    owned.jobs.add(job.id)
    owned.job_history[job.id] = job.status
    cleanup.callback(cleanup_job, client, owned, job.id)
    print('Recorded owned call-job ID:', job.id)
    return job


def outbound_jobs_scenario(client, owned, models, cleanup):
    from azure.core import MatchConditions
    now = datetime.now(timezone.utc)
    scheduled = submit_owned_job(
        client, owned, models, cleanup,
        models.TelephonyCallJobSchedule(not_before=now + timedelta(hours=1), expires_at=now + timedelta(hours=2)),
        'Consented SDK cancellation demonstration; cancel before dispatch.',
    )
    current, etag = client.beta.voice_agents.telephony.get_call_job(
        agent_name=owned.agent_name, call_job_id=scheduled.id, cls=with_etag,
    )
    if current.status not in ('accepted', 'waiting_for_schedule', 'queued', 'waiting_for_retry') or current.attempt_count != 0:
        raise RuntimeError(f'Cancellation demo job is not an undispatched cancellable job: {current.status}, attempts={current.attempt_count}.')
    cancelled = client.beta.voice_agents.telephony.cancel_call_job(
        agent_name=owned.agent_name, call_job_id=current.id, etag=etag, match_condition=MatchConditions.IfNotModified,
    )
    observe_job(owned, cancelled)
    wait_job(client, owned, current.id, {'cancelled'})
    immediate = submit_owned_job(
        client, owned, models, cleanup,
        models.TelephonyCallJobSchedule(expires_at=datetime.now(timezone.utc) + timedelta(minutes=5)),
        'Consented SDK direct outbound test; recipient answers and hangs up promptly.',
    )
    if immediate.id == scheduled.id:
        raise RuntimeError('Distinct idempotency keys unexpectedly resolved to the same job; do not pretend cancellation and calling were separate.')
    wait_job(client, owned, immediate.id, {'completed'})


## 12. Transfer one active inbound call; end a second fresh call

An operator is required, not an unbounded `input()` prompt. Within each bounded window, call the dedicated Twilio number **from `INBOUND_CALLER_NUMBER`**. Calls are discovered only under the new owned agent, matched to the controlled caller/provider number, and recorded immediately. Each management command re-reads the call and requires `in_progress` with an agent-ready/bridging phase. A call that already ended is an error, not an excuse to send a command with a stale ID.

First transfer to the configured named target and require the original call's successful `managed_transfer` termination. Then place a second call; require its successful `managed_hangup` termination after `end_call`. Never end a transferred/terminal original call as if it were still active. Test participants must hang up any **transferred provider leg**, which is not a separately addressable call in this SDK inventory. Cleanup manages only the original owned test-call IDs.

In [ ]:
def observe_call(owned, call):
    owned.call_history[call.id] = call.status
    if call.status in ('success', 'failed'):
        owned.calls.discard(call.id)
    elif call.status != 'in_progress':
        raise RuntimeError(f'Unknown call state {call.status} for {call.id}; do not send a management command.')


def read_active_call(client, owned, call_id):
    if call_id not in owned.calls:
        raise RuntimeError('Refusing to control a call not recorded as an active owned test call.')
    call = client.beta.voice_agents.telephony.get_call(agent_name=owned.agent_name, call_id=call_id)
    observe_call(owned, call)
    if call.status != 'in_progress' or call.phase not in ('agent_session_ready', 'bridging'):
        raise RuntimeError(f'Call {call.id} is no longer ready/active: {call.status}/{call.phase}. Place a fresh test call in a new run.')
    return call


def wait_new_inbound_call(client, owned, cleanup, excluded, purpose):
    print(f'OPERATOR: place a NEW inbound call from the configured controlled caller for {purpose}; wait bound {CALL_WAIT_TIMEOUT_S}s.')
    deadline = time.monotonic() + CALL_WAIT_TIMEOUT_S
    selected = None
    while time.monotonic() < deadline:
        calls = limited_items(client.beta.voice_agents.telephony.list_calls(
            agent_name=owned.agent_name, provider='twilio', limit=100, order='desc',
        ), 'inbound discovery')
        unexpected = {call.id for call in calls if call.status == 'in_progress' and (
            call.caller_number != INBOUND_CALLER_NUMBER or call.provider_number != TELEPHONY_PHONE_NUMBER
        )}
        if unexpected:
            owned.unexpected_calls.update(unexpected)
            raise RuntimeError('Unexpected active caller on the dedicated binding. No control command will be sent; reconcile LAST_LIVE_RUN.')
        candidates = [call for call in calls if call.id not in excluded
                      and call.caller_number == INBOUND_CALLER_NUMBER
                      and call.provider_number == TELEPHONY_PHONE_NUMBER]
        for call in candidates:
            if call.id not in owned.call_history:
                owned.calls.add(call.id)
                owned.call_history[call.id] = call.status
                cleanup.callback(cleanup_call, client, owned, call.id)
        if len(candidates) > 1:
            raise RuntimeError('More than one matching test call arrived. Stop parallel dialing; do not guess which call to transfer.')
        if candidates:
            selected = candidates[0].id
            call = client.beta.voice_agents.telephony.get_call(agent_name=owned.agent_name, call_id=selected)
            observe_call(owned, call)
            if call.status != 'in_progress':
                raise RuntimeError(f'Test call {call.id} ended before control: {call.status}, {call.end_reason}.')
            if call.phase in ('agent_session_ready', 'bridging'):
                return call.id
        pause_before_retry(deadline, f'Inbound {purpose}; check the dedicated webhook, caller ID, and provider connection')
    raise TimeoutError(f'No ready inbound {purpose} call arrived; last observed ID: {selected}.')


def wait_call_terminal(client, owned, call_id, expected_reason=None):
    deadline = time.monotonic() + CALL_WAIT_TIMEOUT_S
    while time.monotonic() < deadline:
        call = client.beta.voice_agents.telephony.get_call(agent_name=owned.agent_name, call_id=call_id)
        observe_call(owned, call)
        if call.status in ('success', 'failed'):
            if call.status != 'success' or (expected_reason is not None and call.end_reason != expected_reason):
                raise RuntimeError(f'Call {call.id} ended as {call.status}, reason={call.end_reason}; expected {expected_reason}.')
            print('Confirmed call termination:', call.id, call.end_reason)
            return call
        pause_before_retry(deadline, f'Call {call_id} termination')
    raise TimeoutError(f'Call {call_id} did not reach a confirmed terminal state.')


def inbound_call_control_scenario(client, owned, cleanup):
    first = wait_new_inbound_call(client, owned, cleanup, set(), 'transfer')
    read_active_call(client, owned, first)
    transferred = client.beta.voice_agents.telephony.transfer_call(
        agent_name=owned.agent_name, call_id=first, target='notebook-controlled-destination',
    )
    observe_call(owned, transferred)
    wait_call_terminal(client, owned, first, expected_reason='managed_transfer')
    print('OPERATOR: hang up the transferred provider leg before making the next test call.')
    second = wait_new_inbound_call(client, owned, cleanup, {first}, 'hangup')
    if second == first:
        raise RuntimeError('Hangup must use a distinct fresh inbound call, never the transferred original.')
    read_active_call(client, owned, second)
    ended = client.beta.voice_agents.telephony.end_call(agent_name=owned.agent_name, call_id=second)
    observe_call(owned, ended)
    wait_call_terminal(client, owned, second, expected_reason='managed_hangup')


## 13. Cleanup is an implemented phase, not an instruction to the reader

Callbacks run in dependency order: settle owned calls/jobs, clear the exact owned transfer set, delete the owned binding with a current ETag, delete finalized owned conversations (cascading their stored items/responses/audio), then delete the exact agent version. Each discovered session conversation ID is registered immediately; a final bounded listing on the **new owned agent only** discovers telephony or partially observed session conversations as well.

A network failure, permission error, concurrency/ownership conflict, unfinalized recording, or still-connected outbound call is **not swallowed**. An unresolved child or unexpected target set prevents dependent agent deletion; unexpected active callers are never controlled by the notebook. Inspect `LAST_LIVE_RUN` for remaining IDs, terminate test phone legs, and recover with the portal/SDK before re-running. The SDK has no call-job or call-record deletion method in this inventory; terminal history is not falsely reported as deleted. No cleanup can run after process/kernel termination or recover an unacknowledged server creation with certainty.

Provider webhook routing is operator-owned and is never edited by this notebook. Remove/restore the temporary dedicated-number webhook yourself. Likewise, cleanup does not delete the pre-existing project connection, provider resource, phone number, or supplied WAV.

In [ ]:
def cleanup_call(client, owned, call_id):
    if call_id not in owned.calls:
        print('Cleanup: call already confirmed terminal:', call_id)
        return
    deadline = time.monotonic() + CALL_WAIT_TIMEOUT_S
    while True:
        call = client.beta.voice_agents.telephony.get_call(agent_name=owned.agent_name, call_id=call_id)
        observe_call(owned, call)
        if call.status == 'failed':
            raise RuntimeError(f'Owned call failed during cleanup: {call_id}, {call.end_reason}.')
        if call.status == 'success':
            print('Cleanup: call is terminal; no stale end_call was sent:', call_id)
            return
        if call.phase in ('agent_session_ready', 'bridging'):
            break
        pause_before_retry(deadline, f'Owned call {call_id} must become controllable or terminal before cleanup')
    print('Cleanup: ending still-active owned inbound test call:', call_id)
    ended = client.beta.voice_agents.telephony.end_call(agent_name=owned.agent_name, call_id=call_id)
    observe_call(owned, ended)
    wait_call_terminal(client, owned, call_id)


def cleanup_job(client, owned, job_id):
    from azure.core import MatchConditions
    if job_id not in owned.jobs:
        print('Cleanup: job already confirmed terminal:', job_id)
        return
    current, etag = client.beta.voice_agents.telephony.get_call_job(
        agent_name=owned.agent_name, call_job_id=job_id, cls=with_etag,
    )
    observe_job(owned, current)
    if current.status in JOB_TERMINAL:
        if current.status not in ('cancelled', 'completed'):
            raise RuntimeError(f'Owned job failed during cleanup: {job_id}, {current.status}, {current.terminal_reason}.')
        print('Cleanup: job is terminal; no redundant cancel was sent:', job_id)
        return
    if current.status != 'cancellation_requested':
        print('Cleanup: requesting cancellation; an already connected call still needs to finish:', job_id)
        cancelled = client.beta.voice_agents.telephony.cancel_call_job(
            agent_name=owned.agent_name, call_job_id=job_id, etag=etag, match_condition=MatchConditions.IfNotModified,
        )
        observe_job(owned, cancelled)
    wait_job(client, owned, job_id, {'cancelled', 'completed'})


def cleanup_transfer_targets(client, owned):
    from azure.core import MatchConditions
    verify_quiet_telephony(client, owned)
    current, etag = client.beta.voice_agents.telephony.get_transfer_targets(agent_name=owned.agent_name, cls=with_etag)
    if target_signature(current.transfer_targets) != owned.target_signature:
        raise RuntimeError('Transfer targets changed outside this run; refusing to clear a different configuration.')
    cleared = client.beta.voice_agents.telephony.replace_transfer_targets(
        agent_name=owned.agent_name, transfer_targets=[], etag=etag, match_condition=MatchConditions.IfNotModified,
    )
    if cleared.transfer_targets:
        raise RuntimeError('Owned transfer targets were not cleared.')
    owned.target_signature = None
    print('Cleanup: owned transfer targets cleared.')


def cleanup_binding(client, owned, binding_id):
    from azure.core import MatchConditions
    verify_quiet_telephony(client, owned)
    if binding_id not in owned.bindings:
        raise RuntimeError('Refusing to delete an unowned binding.')
    binding, etag = client.beta.voice_agents.telephony.get_binding(
        agent_name=owned.agent_name, binding_id=binding_id, cls=with_etag,
    )
    if binding.id != binding_id:
        raise RuntimeError('Binding ID changed during cleanup.')
    client.beta.voice_agents.telephony.delete_binding(
        agent_name=owned.agent_name, binding_id=binding_id, etag=etag, match_condition=MatchConditions.IfNotModified,
    )
    owned.bindings.remove(binding_id)
    print('Cleanup: deleted owned binding:', binding_id)


def delete_owned_conversation(client, owned, conversation_id):
    if conversation_id not in owned.conversations:
        raise RuntimeError('Refusing to delete an unowned conversation.')
    conversation = wait_persisted(
        f'Cleanup finalization for {conversation_id}',
        lambda: client.beta.voice_agents.conversations.get(agent_name=owned.agent_name, conversation_id=conversation_id),
        lambda value: conversation_finished(value, for_cleanup=True),
    )
    client.beta.voice_agents.conversations.delete(agent_name=owned.agent_name, conversation_id=conversation_id)
    owned.conversations.remove(conversation_id)
    owned.deleted_conversations.add(conversation_id)
    print('Cleanup: deleted owned conversation and its stored data:', conversation_id)
    if conversation.status == 'failed':
        raise RuntimeError(f'Owned conversation was deleted, but its persistence failed: {conversation.last_error}')


def cleanup_conversations(client, owned):
    require_no_active_telephony(owned)
    if owned.bindings:
        raise RuntimeError('Binding cleanup is unresolved; retain conversation data until telephony ingress is closed.')
    discovered = limited_items(client.beta.voice_agents.conversations.list(agent_name=owned.agent_name, limit=100), 'owned cleanup discovery')
    for conversation in discovered:
        if conversation.id not in owned.deleted_conversations:
            owned.conversations.add(conversation.id)
    owned.conversation_discovery_complete = True
    with ExitStack() as deletions:
        for conversation_id in sorted(owned.conversations):
            deletions.callback(delete_owned_conversation, client, owned, conversation_id)


def cleanup_agent(client, owned):
    require_no_active_telephony(owned)
    if (not owned.conversation_discovery_complete or owned.conversations
            or owned.bindings or owned.target_signature is not None or owned.ownership_conflicts):
        raise RuntimeError(
            f'Retaining agent {owned.agent_name}/{owned.agent_version}: child cleanup is unresolved. Inspect LAST_LIVE_RUN.'
        )
    if owned.agent_version is None:
        raise RuntimeError('No created agent version was recorded; refusing a broad agent deletion.')
    deleted = client.agents.delete_version(agent_name=owned.agent_name, agent_version=owned.agent_version)
    if not deleted.deleted or deleted.name != owned.agent_name or deleted.version != owned.agent_version:
        raise RuntimeError('The service did not confirm deletion of the exact owned agent version; inspect LAST_LIVE_RUN.')
    print('Cleanup: deleted the exact owned agent version:', owned.agent_name, owned.agent_version)
    owned.agent_version = None


## 14. Run the selected scenarios

Run the notebook top to bottom. **Only this cell calls the orchestrator.** Leave all flags false for a standard-library-only tour. For a live run, finish the prerequisites in section 3, change only the desired flags/consents/configuration, then rerun the configuration and this cell.

The orchestrator prints enabled/disabled branches explicitly. The master guard precedes version inspection, SDK imports, environment access, file reads, authentication, or resource creation. HTTP timeouts and disabled automatic retries keep service failures visible; outbound creation also carries its documented idempotency key. A successful final message is printed only after cleanup returns without error.

In [ ]:
def run_notebook():
    global LAST_LIVE_RUN
    validate_branch_dependencies()
    if not RUN_LIVE_DEMO:
        print('OFFLINE: RUN_LIVE_DEMO=False. No SDK imports, environment/credential checks, network, media files, or resource changes.')
        print('DISABLED: base live agent, realtime text, persisted conversation/audio reads, and service cleanup.')
        for label in branch_flags():
            print('DISABLED:', label)
        print('All 45 methods have implemented opt-in scenarios; none were executed against Azure in this offline run.')
        return
    for label, enabled in branch_flags().items():
        print('ENABLED:' if enabled else 'DISABLED:', label)
    endpoint, models, pcm = preflight_live()
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential
    owned = OwnedRun(agent_name='notebook-voice-' + uuid4().hex)
    LAST_LIVE_RUN = owned
    with DefaultAzureCredential(process_timeout=HTTP_TIMEOUT_S) as credential:
        with AIProjectClient(
            endpoint=endpoint, credential=credential, allow_preview=True,
            connection_timeout=HTTP_TIMEOUT_S, read_timeout=HTTP_TIMEOUT_S, retry_total=0,
        ) as client:
            with ExitStack() as cleanup:
                create_owned_agent(client, owned, models, cleanup)
                state = text_and_optional_audio_scenario(client, owned, models, pcm)
                persisted_conversation_scenario(client, owned, state)
                if RUN_WEBRTC_AUDIO:
                    webrtc_audio_scenario(client, owned, models)
                if RUN_AVATAR:
                    avatar_scenario(client, owned, models)
                if RUN_TELEPHONY:
                    telephony_configuration_scenario(client, owned, models, cleanup)
                    if RUN_OUTBOUND_CALLS:
                        outbound_jobs_scenario(client, owned, models, cleanup)
                    if RUN_INBOUND_CALL_CONTROL:
                        inbound_call_control_scenario(client, owned, cleanup)
    print('Selected live scenarios and owned-resource cleanup completed without an error.')
    if RUN_INBOUND_CALL_CONTROL:
        print('OPERATOR REMINDER: restore/remove the dedicated-number provider webhook and hang up any transferred leg.')
    print('Ownership/recovery ledger remains in LAST_LIVE_RUN; terminal provider/job history is not deleted by this API.')


run_notebook()


## 15. Pinned sources and service-dependent limits

Azure source and sample links are fixed to **`azure-ai-projects_2.7.0`** (released 2026-09-18):

* [Generated operations](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/operations/_operations.py): conversation routes, generated-vs-heard audio, BYOS/recording errors, telephony request parameters, ETags, inbound-only management, cancellation semantics.
* [Typed models](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_models.py) and [enums](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_enums.py): precise request fields, event IDs, PCM payloads, SDP fields, durations as `timedelta`, job/call terminal states.
* [Handwritten sync realtime client](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_realtime.py): all returned manager/connection/resource helpers, typed `send`, receive timeout, decoded audio bytes, and close behavior.
* [Model mapping implementation](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_utils/model_base.py): explicit null for disabling VAD without inventing fields.
* [Release dependency/voice-extra declarations](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/pyproject.toml); [websockets 15.0.1 sync transport parameters](https://github.com/python-websockets/websockets/blob/15.0.1/src/websockets/sync/client.py) used via the SDK's documented forwarding of connection kwargs.
* [Basic voice agent lifecycle](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/agents/voice/sample_voice_agent_basic.py).
* [Live typed text conversation](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/agents/voice/sample_voice_agent_live_text_conversation.py).
* [Live audio conversation, async](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/agents/voice/sample_voice_agent_live_audio_conversation_async.py). This notebook instead reads a bounded user recording; it does not copy the sample's endless microphone loop or optional silent playback fallback.
* [Read conversation audio](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/agents/voice/sample_voice_agent_read_conversation_audio.py). This notebook uses bounded memory rather than writing WAVs, and raises rather than silently skipping unavailable artifacts.
* [Voice configuration/tools sample](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/agents/voice/sample_voice_agent_with_tools.py).
* Optional adapter source: aiortc **1.14.0** [peer connection](https://github.com/aiortc/aiortc/blob/1.14.0/src/aiortc/rtcpeerconnection.py), [ICE configuration](https://github.com/aiortc/aiortc/blob/1.14.0/src/aiortc/rtcconfiguration.py), and [media example](https://github.com/aiortc/aiortc/blob/1.14.0/examples/server/server.py).

**Not established by static verification:** your tenant's preview permissions; model/voice/avatar availability; actual persistence latency; whether interrupted generated audio is retained for your selected model; current service WebRTC/avatar support; network/codec/ICE reachability; provider routing and ETag behavior; cancellation race outcomes; or successful billable calls/transfers. These require the corresponding explicitly opted-in service run. This notebook does not claim live validation.

In particular, cancellation can lose a race to normal completion, generated-audio artifacts can remain unavailable, and a connected outbound call may outlive a cancellation request. Such cases deliberately fail with actionable context, execute the available cleanup, and preserve unresolved ownership information rather than fabricate method coverage or resource deletion.

## 16. Complete release-source invocation audit

This appendix records **every distinct provided-argument shape**, not the full SDK signatures or extra runnable examples. Repeated call sites are coalesced. In the notation below, `name=` means a keyword argument and `pos` means a positional argument; these are audit records, not placeholder implementations. The implemented calls remain in the preceding code cells.

Static checks covered **67 SDK method call sites** (64 Voice Agents calls plus 3 supporting agent calls), **48 distinct SDK methods** (the required 45 plus the 3 supporting agent methods), and **24 model-constructor call sites using 19 public model classes**. Required arguments, keyword names, positional binding, overload alternatives, enum members, and all 39 referenced public model/event/enum exports were checked against the release-tagged sources. This is source validation, not Python execution or service validation.

### 16.1 Conversation operations: all 14 methods

Shapes match `BetaVoiceAgentsConversationsOperations` in `operations/_operations.py`. The three `raw_response_hook` keywords are Azure Core pipeline controls, not JSON request fields.

```text
client.beta.voice_agents.conversations.delete(agent_name=, conversation_id=)
client.beta.voice_agents.conversations.download_audio(agent_name=, conversation_id=, raw_response_hook=)
client.beta.voice_agents.conversations.download_audio_item(agent_name=, conversation_id=, item_id=, raw_response_hook=)
client.beta.voice_agents.conversations.download_generated_audio_item(agent_name=, conversation_id=, item_id=, raw_response_hook=)
client.beta.voice_agents.conversations.get(agent_name=, conversation_id=)
client.beta.voice_agents.conversations.get_audio(agent_name=, conversation_id=)
client.beta.voice_agents.conversations.get_audio_item(agent_name=, conversation_id=, item_id=)
client.beta.voice_agents.conversations.get_generated_audio_item(agent_name=, conversation_id=, item_id=)
client.beta.voice_agents.conversations.get_item(agent_name=, conversation_id=, item_id=)
client.beta.voice_agents.conversations.get_response(agent_name=, conversation_id=, response_id=)
client.beta.voice_agents.conversations.list(agent_name=, limit=)
client.beta.voice_agents.conversations.list_items(agent_name=, conversation_id=, limit=, order=)
client.beta.voice_agents.conversations.list_response_items(agent_name=, conversation_id=, response_id=, limit=, order=)
client.beta.voice_agents.conversations.list_responses(agent_name=, conversation_id=, limit=, order=)
```

### 16.2 Realtime entry point and every returned helper

All these definitions are in the handwritten `_realtime.py`, not generated operations. `connect` forwards `open_timeout`, `close_timeout`, and `max_size` through `**kwargs` to `websockets.sync.client.connect`. The two positional `conn.send` payload types used here are `RealtimeClientEventConversationItemRetrieve` and `VoiceAgentClientEventRtcCallSdpCreate`.

```text
client.beta.voice_agents.realtime.connect(agent_name=, open_timeout=, close_timeout=, max_size=)
manager.enter()
conn.close(code=, reason=)
conn.recv(timeout=)
conn.send(pos)
conn.session.avatar_connect(client_sdp=)
conn.session.update(session=)
conn.input_audio_buffer.append(audio=)
conn.input_audio_buffer.clear()
conn.input_audio_buffer.commit()
conn.output_audio_buffer.clear()
conn.conversation.item.create(item=)
conn.conversation.item.delete(item_id=)
conn.conversation.item.retrieve(item_id=)
conn.conversation.item.truncate(item_id=, content_index=, audio_end_ms=)
conn.response.cancel(response_id=)
conn.response.create(response=)
```

### 16.3 Telephony: all 14 methods, including every keyword variant

Shapes match `BetaVoiceAgentsTelephonyOperations` in `operations/_operations.py`. `cls` is the generated response callback, invoked with `(pipeline_response, deserialized_result, response_headers)`; this notebook deliberately returns `(model, ETag)` from it. `transfer_call(target=...)` and `replace_transfer_targets(transfer_targets=...)` use the actual flattened keyword overloads; their optional `_Unset` body is not a missing request. Conditional mutations supply both `etag` and `match_condition`.

```text
client.beta.voice_agents.telephony.cancel_call_job(agent_name=, call_job_id=, etag=, match_condition=)
client.beta.voice_agents.telephony.create_binding(agent_name=, telephony_binding=)
client.beta.voice_agents.telephony.create_call_job(agent_name=, idempotency_key=, body=)
client.beta.voice_agents.telephony.delete_binding(agent_name=, binding_id=, etag=, match_condition=)
client.beta.voice_agents.telephony.end_call(agent_name=, call_id=)
client.beta.voice_agents.telephony.get_binding(agent_name=, binding_id=, cls=)
client.beta.voice_agents.telephony.get_call(agent_name=, call_id=)
client.beta.voice_agents.telephony.get_call_job(agent_name=, call_job_id=)
client.beta.voice_agents.telephony.get_call_job(agent_name=, call_job_id=, cls=)
client.beta.voice_agents.telephony.get_transfer_targets(agent_name=, cls=)
client.beta.voice_agents.telephony.list_bindings(agent_name=, limit=)
client.beta.voice_agents.telephony.list_calls(agent_name=, limit=)
client.beta.voice_agents.telephony.list_calls(agent_name=, status=, limit=)
client.beta.voice_agents.telephony.list_calls(agent_name=, provider=, limit=, order=)
client.beta.voice_agents.telephony.replace_transfer_targets(agent_name=, transfer_targets=, etag=, match_condition=)
client.beta.voice_agents.telephony.transfer_call(agent_name=, call_id=, target=)
client.beta.voice_agents.telephony.update_binding(agent_name=, binding_id=, body=, etag=, match_condition=)
```

### 16.4 All 19 SDK model constructors (23 distinct keyword shapes)

All constructors are exported by `models/__init__.py` and match typed overloads in `models/_models.py`. None of these 19 classes is replaced by `models/_patch.py`. This includes inherited required fields such as `RealtimeConversationItemMessageUser.type`; discriminator defaults are not invented request fields.

```text
models.CreateTelephonyCallJobRequest(destination=, connection_name=, source=, purpose=, schedule=)
models.CreateTwilioTelephonyBindingRequest(connection_name=, phone_number=, label=)
models.PSTNTelephonyTransferDestination(value=)
models.RealtimeAudioFormatsAudioPcm(rate=)
models.RealtimeClientEventConversationItemRetrieve(item_id=)
models.RealtimeConversationItemMessageUser(type=, content=)
models.RealtimeConversationItemMessageUserContent(type=, text=)
models.TelephonyCallJobSchedule(not_before=, expires_at=)
models.TelephonyCallJobSchedule(expires_at=)
models.TelephonyOutboundDestination(type=, value=)
models.TelephonyTransferTarget(name=, description=, destination=)
models.UpdateTelephonyBindingRequest(label=)
models.VoiceAgentAudioConfig(output=)
models.VoiceAgentAudioConfig(input=)
models.VoiceAgentAudioInputConfig(format=)
models.VoiceAgentAudioOutputConfig(format=, voice=, voice_type=)
models.VoiceAgentClientEventRtcCallSdpCreate(sdp_offer=)
models.VoiceAgentDefinition(model_type=, model=, instructions=, audio=, output_modalities=, store=)
models.VoiceAgentResponseCreateParams(instructions=, max_output_tokens=)
models.VoiceAgentSessionAvatarConfig(type=, character=, style=, output_protocol=)
models.VoiceAgentSessionUpdateConfig(instructions=, max_output_tokens=)
models.VoiceAgentSessionUpdateConfig(audio=)
models.VoiceAgentSessionUpdateConfig(avatar=, output_modalities=)
```

The only model mutation outside constructors is `input_config['turn_detection'] = None`: the key is a declared `VoiceAgentAudioInputConfig` field, and `_utils/model_base.py` implements the explicit mapping assignment. Model/event `get(key)` and `get(key, default)` calls use that same public mapping implementation. Typed audio deltas are already decoded `bytes`; persisted audio durations are `datetime.timedelta`, not guessed integer milliseconds.

### 16.5 Supporting client/auth/agent setup and HTTP cleanup

```text
AIProjectClient(endpoint=, credential=, allow_preview=, connection_timeout=, read_timeout=, retry_total=)
DefaultAzureCredential(process_timeout=)
client.agents.get(agent_name=)
client.agents.create_version(agent_name=, definition=)
client.agents.delete_version(agent_name=, agent_version=)
raw_http_response.close()  [registered as an ExitStack callback]
```

`AIProjectClient` is the `_patch.py` export, which delegates these constructor arguments to `_client.py` and `_configuration.py`. HTTP timeout and retry options are forwarded Azure Core configuration/policy arguments. The patch in `operations/_patch_agents.py` overrides `create_version`, accepts the provided `definition` keyword, and adds the Voice Agents preview header when `allow_preview=True`; `get` and `delete_version` inherit their generated shapes. `DeleteAgentVersionResponse.deleted/name/version` are checked before recording deletion success.

The credential parameter is documented and consumed by `DefaultAzureCredential.__init__`. `raw_response_hook` receives one `PipelineResponse`, while `cls` receives three arguments: do not conflate these callback shapes. The raw synchronous `azure.core.rest.HttpResponse.close()` takes no arguments. Client and credential context managers provide their ordinary SDK cleanup in addition to the explicit resource callbacks.

### 16.6 Every optional aiortc constructor and media method

These are separate dependency APIs, not additional Voice Agents methods. Checked against the pinned aiortc 1.14.0 sources and its official event-handler example:

```text
RTCConfiguration(iceServers=)
RTCIceServer(**server)  [keys: urls; optional username, credential]
RTCPeerConnection(pos)  [one RTCConfiguration]
RTCSessionDescription(sdp=, type=)  [type is 'answer']
pc.on(pos)  [the 'track' event decorator]
pc.addTransceiver(pos, direction=)  [kind is audio/video; direction is recvonly]
pc.createOffer()
pc.setLocalDescription(pos)  [the genuine locally created offer]
pc.setRemoteDescription(pos)  [the genuine server answer]
pc.close()
remote_track.recv()
```

Besides the section 15 links, see [RTCSessionDescription](https://github.com/aiortc/aiortc/blob/1.14.0/src/aiortc/rtcsessiondescription.py) and [RemoteStreamTrack.recv](https://github.com/aiortc/aiortc/blob/1.14.0/src/aiortc/rtcrtpreceiver.py). The latter raises `MediaStreamError` for ended tracks; the notebook does not convert that into an empty successful frame.

### 16.7 Extra source paths for automated signature validation

The shared generated operations/models/enums and handwritten realtime files are necessary but not sufficient for public-export/forwarding checks. Under the same `azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/` release base, also inspect:

| Release-relative source | Why it matters |
| --- | --- |
| [azure/ai/projects/_patch.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_patch.py) | Public patched client constructor, keyword forwarding. |
| [azure/ai/projects/_client.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_client.py) | Generated constructor and PipelineClient wiring. |
| [azure/ai/projects/_configuration.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_configuration.py) | Azure Core retry/hook policy configuration. |
| [azure/ai/projects/operations/_patch.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/operations/_patch.py) | Injects handwritten realtime; wraps nested voice groups with the preview-header proxy; exports returned realtime types. |
| [azure/ai/projects/operations/_patch_agents.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/operations/_patch_agents.py) | Actual create_version overloads and preview forwarding for supporting setup. |
| [azure/ai/projects/models/_patch.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_patch.py) | Voice preview-header constants; confirms no used voice constructor overrides. |
| [azure/ai/projects/models/__init__.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/__init__.py) | Public constructor/event/enum exports. |
| [azure/ai/projects/__init__.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/__init__.py) | Public client export replaced by the patch. |
| [azure/ai/projects/_utils/model_base.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/_utils/model_base.py) | MutableMapping methods, explicit null, serialization/deserialization. |

Supporting packages at the same Azure repository tag: [identity/_credentials/default.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/identity/azure-identity/azure/identity/_credentials/default.py), [core/configuration.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/core/azure-core/azure/core/configuration.py), [core/pipeline/policies/_retry.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/core/azure-core/azure/core/pipeline/policies/_retry.py), [core/pipeline/policies/_custom_hook.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/core/azure-core/azure/core/pipeline/policies/_custom_hook.py), and [core/rest/_rest_py3.py](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/core/azure-core/azure/core/rest/_rest_py3.py). These establish the supporting argument/callback contracts; they are not assertions that the Projects package pins those packages to the monorepo commit's versions.

No installed Azure modules were imported to perform this audit. The executable call spelling, default-off guards, existing cell IDs, and `cosmopilot.modules` metadata were preserved when adding this appendix.